# Prompt Engineering 101 - Part VI.
## Agents

----

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini + Game Engine)
# We are installing the Google GenAI SDK.

!pip install -q -U google-genai

from google.colab import userdata
from google import genai
from IPython.display import display, Markdown
import time

# 2. Configure the API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 3. Create Wrapper Class (Same as Mod 4)
class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.5-flash'):
        self.client = genai.Client(api_key=api_key)
        
        # Priority list of models and their availability status
        # True = Available, False = Exhausted (429)
        self.models = {
            'gemini-2.5-flash': True,
            'gemini-2.5-flash-lite': True,
            'gemini-3-flash-preview': True,
        }
        
        # Validation: Ensure the user's choice is valid
        if preferred_model not in self.models:
            raise ValueError(f"Invalid model '{preferred_model}'. "
                             f"Valid options: {list(self.models.keys())}")
            
        self.current_model = preferred_model

    def _get_next_available_model(self):
        """Sets the first model from the non-exhausted models as the currently selected model. 
        Raises error if no available model left."""
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
    
        raise RuntimeError("❌ All available models are currently exhausted. "
                           "Please wait for quotas to reset.")

    def generate_content(self, contents, config=None):
        """
        Recursively attempts to generate content.
        If a model fails with a quota error, it marks it as unavailable 
        and retries with the next available model.
        """
        try:
            # Attempt generation
            response = self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
            return response

        except Exception as e:
            # Check for Rate Limit / Quota errors
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Marking as unavailable...")
                
                # Update State: Mark current as failed
                self.models[self.current_model] = False
                
                # Switch to next available
                self._get_next_available_model()
            
                # Recursive Step: Try again with the new model
                return self.generate_content(contents, config)
            
            # If it's a logic error (e.g. invalid prompt), raise immediately
            raise e

try:
    model = GeminiModel(GEMINI_API_KEY, preferred_model='gemini-2.5-flash')
    print(f"✅ Connection Established. Using {model.current_model}.")
except Exception as e:
    print(f"❌ Error: {e}")


# 4. The Text Adventure Engine (Hidden Complexity)
class TextAdventure:
    def __init__(self):
        self.state = "You are in a stone corridor. North is a locked door. South is a dark pit. There is a brass key on the floor."
        self.inventory = []
        self.log = []
        
    def step(self, action):
        action = action.lower().strip()
        observation = ""
        
        if "pick up" in action and "key" in action:
            if "key" not in self.inventory:
                self.inventory.append("key")
                observation = "You picked up the brass key."
            else:
                observation = "You already have the key."
        elif "open" in action and "door" in action:
            if "key" in self.inventory:
                observation = "VICTORY! The door opens and you escape!"
                return observation, True
            else:
                observation = "The door is locked."
        elif "look" in action:
            observation = self.state
        else:
            observation = "Nothing happens."
            
        return observation, False

----

# 🏠 Homework: The Customer Support Manager

### The Scenario
You are building the "Brain" for a Customer Support center.
You need a Multi-Agent system to handle incoming emails.

### The Task
Write a Python script that simulates this workflow:
1.  **The Router:** Analyzes an incoming email and decides if it is "Technical Support" or "Billing".
2.  **The Specialist:**
    * If Technical: "Ask for the error code."
    * If Billing: "Ask for the invoice number."
3.  **The Safety Check:** If the user seems angry (detect sentiment), flag for "Human Review".

### Submission
Submit the Python code that chains these 3 prompts together.

In [ ]:
# @title (Hidden) Answer Key
email = "I am furious! My card was charged twice!"

# 1. Sentiment Check
sentiment = model.generate_content(f"Analyze sentiment of: '{email}'. Return 'Angry' or 'Neutral'.").text
if "Angry" in sentiment:
    print("🚨 FLAG FOR HUMAN REVIEW")
else:
    # 2. Router
    category = model.generate_content(f"Classify '{email}' as 'Tech' or 'Billing'.").text
    
    # 3. Specialist
    if "Billing" in category:
        print(model.generate_content("Write a polite reply asking for the Invoice Number.").text)
    else:
        print(model.generate_content("Write a polite reply asking for the Error Code.").text)
